# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.35437657  0.6789758   0.41324583 -0.0736165  -0.23300091]
 [-0.29563907  0.76523352 -0.42998814 -0.76764828  0.27749554]
 [ 0.52822225  0.88087218 -0.89468069 -0.85864177  0.04473372]
 [ 0.62271222 -0.3041627  -0.97688279 -0.78709298  0.8340052 ]
 [-0.47348331  0.52920354  0.56431592  0.10846498  0.23060267]
 [ 0.30027234 -0.23593725  0.53092187 -0.53893267  0.64707334]
 [ 0.14382805  0.70975484 -0.12438782  0.80436757 -0.36299706]
 [ 0.57240804  0.92678105  0.77262745 -0.33933796  0.0314875 ]
 [-0.08312164 -0.39083114  0.21980398  0.4348023   0.20968637]
 [-0.05410415  0.45541943 -0.50686754  0.08954633 -0.84557215]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a2', 'a2', 'a1', 'a2', 'a2', 'a2', 'a1', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 1, 0, 1, 0, 1, 0, 0, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:00<00:28,  1.16it/s]

SVI:   3%|▎         | 1/34 [00:00<00:28,  1.16it/s, loss=1788.6613]

SVI:   6%|▌         | 2/34 [00:00<00:27,  1.16it/s, loss=1908.4496]

SVI:   9%|▉         | 3/34 [00:00<00:26,  1.16it/s, loss=2121.2209]

...

SVI:  84%|████████▍ | 21/25 [00:01<00:03,  1.02it/s, loss=2523.4185]

SVI:  88%|████████▊ | 22/25 [00:01<00:02,  1.02it/s, loss=2387.4241]

SVI:  92%|█████████▏| 23/25 [00:01<00:01,  1.02it/s, loss=2999.5667]

SVI:  96%|█████████▌| 24/25 [00:01<00:00,  1.02it/s, loss=2827.7205]

SVI: 100%|██████████| 25/25 [00:01<00:00,  1.02it/s, loss=2430.0010]